# Behavioral Drift Detection in Distributed Systems
**Dataset:** Google Borg Cluster Traces (313MB, 1.3M rows)  
**Goal:** Detect behavioral drift in CPU and memory metrics across time windows using statistical tests with thresholds learned dynamically from early window behavior.

## Setup

In [ ]:
!pip install pyspark -q

import warnings
warnings.filterwarnings('ignore')

print("Dependencies installed.")

## Data Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify file is accessible
import os
path = "/content/drive/MyDrive/borg_traces_data.csv"
print("File exists:", os.path.exists(path))
print("File size (MB):", round(os.path.getsize(path) / (1024*1024), 1))

## Spark Initialization

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BehavioralDriftDetection") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

## Raw Ingestion

In [ ]:
from pyspark.sql.functions import col

df_raw = spark.read.csv(
    "/content/drive/MyDrive/borg_traces_data.csv",
    header=True,
    inferSchema=True
)

print("Total rows:", df_raw.count())
print("Total columns:", len(df_raw.columns))
print("\nSchema:")
df_raw.printSchema()
df_raw.show(3, truncate=True)

## Preprocessing

In [ ]:
df_raw = spark.read.csv(
    "/content/drive/MyDrive/borg_traces_data.csv",
    header=True,
    inferSchema=False,  # read everything as string first
    multiLine=True,
    escape='"',
    quote='"'
)

print("Total rows:", df_raw.count())
print("Columns:", df_raw.columns)
df_raw.select("time", "average_usage", "maximum_usage").show(5, truncate=False)

## Feature Extraction


In [ ]:
from pyspark.sql.functions import col, regexp_extract, trim, when, lit
from pyspark.sql import functions as F

def extract_struct_field(column_name, field):
    """Extract a field from struct-like string, return null if no match"""
    pattern = f"'{field}':\\s*([\\d.eE+\\-]+)"
    extracted = regexp_extract(col(column_name), pattern, 1)
    return when(extracted == "", lit(None)).otherwise(extracted).cast("double")

def safe_double(column_name):
    """Cast to double, treating empty strings as null"""
    cleaned = trim(col(column_name))
    return when(cleaned == "", lit(None)).otherwise(cleaned).cast("double")

df = df_raw.select(
    safe_double("time").alias("time"),
    extract_struct_field("average_usage", "cpus").alias("avg_cpu_usage"),
    extract_struct_field("average_usage", "memory").alias("avg_memory_usage"),
    extract_struct_field("maximum_usage", "cpus").alias("max_cpu_usage"),
    extract_struct_field("maximum_usage", "memory").alias("max_memory_usage"),
    extract_struct_field("resource_request", "cpus").alias("requested_cpu"),
    extract_struct_field("resource_request", "memory").alias("requested_memory"),
    safe_double("assigned_memory").alias("assigned_memory"),
    safe_double("cycles_per_instruction").alias("cycles_per_instruction"),
    safe_double("memory_accesses_per_instruction").alias("memory_accesses_per_instruction"),
    when(trim(col("scheduling_class")) == "", lit(None))
        .otherwise(trim(col("scheduling_class"))).cast("int").alias("scheduling_class"),
    when(trim(col("failed")) == "", lit(None))
        .otherwise(trim(col("failed"))).alias("failed")
)

df = df.filter(col("time") > 0)
df = df.dropna(subset=["time", "avg_cpu_usage", "avg_memory_usage"])
df.cache()

print("Cleaned rows:", df.count())
df.show(10)
df.printSchema()

## Exploratory Statistics

In [ ]:
# Convert to pandas for stats — Spark used for all heavy lifting above
df_pd = df.toPandas()

print("=== Basic Statistics ===")
print(df_pd[["avg_cpu_usage", "avg_memory_usage", "max_cpu_usage",
             "max_memory_usage", "assigned_memory"]].describe())

print("\n=== Failure Rate ===")
print(df_pd["failed"].value_counts())

print("\n=== Scheduling Class Distribution ===")
print(df_pd["scheduling_class"].value_counts().sort_index())

print("\n=== Total rows:", len(df_pd))

## Time-Based Windowing

In [ ]:
from pyspark.sql.functions import ntile, percent_rank
from pyspark.sql.window import Window

# Sort by time and divide into 10 equal windows
window_spec = Window.orderBy("time")
df = df.withColumn("time_window", ntile(10).over(window_spec))

# Verify window distribution
print("=== Rows per Window ===")
df.groupBy("time_window").count().orderBy("time_window").show()

# Cache since we'll use this repeatedly
df.cache()
print("DataFrame cached.")

## Per-Window Statistics

In [ ]:
from pyspark.sql.functions import mean, stddev, skewness, kurtosis, percentile_approx

metrics = ["avg_cpu_usage", "avg_memory_usage", "max_cpu_usage",
           "max_memory_usage", "assigned_memory"]

# Compute rich statistics per window — all in Spark
agg_exprs = []
for m in metrics:
    agg_exprs += [
        mean(col(m)).alias(f"{m}_mean"),
        stddev(col(m)).alias(f"{m}_std"),
        skewness(col(m)).alias(f"{m}_skew"),
        kurtosis(col(m)).alias(f"{m}_kurtosis"),
        percentile_approx(col(m), 0.25).alias(f"{m}_p25"),
        percentile_approx(col(m), 0.75).alias(f"{m}_p75"),
    ]

window_stats = df.groupBy("time_window").agg(*agg_exprs).orderBy("time_window")
window_stats.show(truncate=False)

# Convert to pandas for visualization
window_stats_pd = window_stats.toPandas()

## Drift Detection using KS Test

In [ ]:
from scipy import stats
import pandas as pd
import numpy as np

# Collect each window as pandas
windows = {}
for w in range(1, 11):
    windows[w] = df.filter(col("time_window") == w).select(metrics).toPandas()
    print(f"Window {w} collected: {len(windows[w])} rows")

# KS test between window 1 (baseline) and all subsequent windows
ks_results = []

for w in range(2, 11):
    for metric in metrics:
        baseline = windows[1][metric].dropna()
        current = windows[w][metric].dropna()

        stat, pvalue = stats.ks_2samp(baseline, current)

        ks_results.append({
            "window": w,
            "metric": metric,
            "ks_statistic": round(stat, 4),
            "p_value": round(pvalue, 6),
            "drift_detected": (pvalue < 0.05) and (stat > 0.20)
        })

ks_df = pd.DataFrame(ks_results)
print("\n=== KS Test Results ===")
print(ks_df[ks_df["drift_detected"] == True])

## Drift Detection using PSI

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

def compute_psi(expected, actual, bins=10):
    min_val = float(np.minimum(expected.min(), actual.min()))
    max_val = float(np.maximum(expected.max(), actual.max()))
    bin_edges = np.linspace(min_val, max_val, bins + 1)

    expected_counts = np.histogram(expected, bins=bin_edges)[0]
    actual_counts = np.histogram(actual, bins=bin_edges)[0]

    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)

    expected_pct = np.where(expected_pct == 0, 1e-4, expected_pct)
    actual_pct = np.where(actual_pct == 0, 1e-4, actual_pct)

    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return round(float(psi), 4)

In [ ]:
metrics = ["avg_cpu_usage", "avg_memory_usage", "max_cpu_usage",
           "max_memory_usage", "assigned_memory"]

# --- Step 1: Compute PSI for ALL windows first ---
psi_results = []
for w in range(2, 11):
    for metric in metrics:
        baseline = windows[1][metric].dropna().values
        current = windows[w][metric].dropna().values
        psi = compute_psi(baseline, current)
        psi_results.append({
            "window": w,
            "metric": metric,
            "psi": psi
        })

psi_df = pd.DataFrame(psi_results)

# --- Step 2: Learn thresholds from warm-up windows (2 and 3) ---
# These are early windows where system is still "normal"
# Threshold = mean PSI + 2 * std PSI observed during warm-up
warmup_psi = psi_df[psi_df["window"].isin([2, 3])]

dynamic_thresholds = {}
for metric in metrics:
    metric_warmup = warmup_psi[warmup_psi["metric"] == metric]["psi"].values
    mean_psi = np.mean(metric_warmup)
    std_psi = np.std(metric_warmup)
    # slight = mean + 1 std, significant = mean + 2 std
    dynamic_thresholds[metric] = {
        "slight": round(mean_psi + 1 * std_psi, 4),
        "significant": round(mean_psi + 2 * std_psi, 4)
    }

print("=== Learned Drift Thresholds (from warm-up windows 2-3) ===")
for metric, thresholds in dynamic_thresholds.items():
    print(f"  {metric}: slight={thresholds['slight']}, significant={thresholds['significant']}")

# --- Step 3: Apply learned thresholds to classify drift ---
def classify_drift(psi, metric):
    slight_thresh = dynamic_thresholds[metric]["slight"]
    sig_thresh = dynamic_thresholds[metric]["significant"]
    if psi >= sig_thresh:
        return "significant"
    elif psi >= slight_thresh:
        return "slight"
    else:
        return "none"

psi_df["drift_level"] = psi_df.apply(
    lambda row: classify_drift(row["psi"], row["metric"]), axis=1
)

print("\n=== PSI Results with Data-Driven Thresholds ===")
print(psi_df)
print("\nDrift level counts:", psi_df["drift_level"].value_counts())

In [ ]:
#PSI-Based Drift Timeline (primary signal)

print("=== DRIFT DETECTED (PSI-based) ===")
psi_drift = psi_df[psi_df['drift_level'].isin(['slight', 'significant'])]
if psi_drift.empty:
    print("No meaningful drift detected.")
else:
    print(psi_drift[['window', 'metric', 'psi', 'drift_level']].to_string(index=False))

print("\n=== DRIFT TIMELINE (metrics drifted per window) ===")
drift_count = psi_df[psi_df['drift_level'] != 'none'].groupby('window').size()
print(drift_count.to_string())

print("\n=== MOST AFFECTED METRICS (mean PSI across windows) ===")
print(psi_df.groupby('metric')['psi'].mean().sort_values(ascending=False).to_string())

print("\n=== STABLE WINDOWS (zero PSI drift) ===")
stable_windows = psi_df.groupby('window').filter(
    lambda x: (x['drift_level'] == 'none').all()
)['window'].unique()
print("Windows with no drift:", sorted(stable_windows))

## Failure Rate Analysis

In [ ]:
import matplotlib.pyplot as plt

#Failure Rate vs Drift Analysis

df_pd = df.toPandas()

failure_rate = df_pd.groupby('time_window')['failed'].apply(
    lambda x: (x.astype(int) == 1).mean() * 100
).reset_index()
failure_rate.columns = ['window', 'failure_rate_pct']

print("=== Failure Rate per Window ===")
print(failure_rate.to_string(index=False))

# Plot PSI of assigned_memory (highest drift metric) vs failure rate
fig, ax1 = plt.subplots(figsize=(10, 4))

assigned_psi = psi_df[psi_df['metric'] == 'assigned_memory'].copy()
# window 1 has no PSI (it's the baseline), add it as 0 for plotting
baseline_row = pd.DataFrame([{'window': 1, 'metric': 'assigned_memory', 'psi': 0.0, 'drift_level': 'none'}])
assigned_psi = pd.concat([baseline_row, assigned_psi], ignore_index=True)

ax1.bar(assigned_psi['window'], assigned_psi['psi'],
        alpha=0.6, color='steelblue', label='PSI — assigned_memory')
ax1.axhline(y=0.1, color='orange', linestyle='--', alpha=0.6, label='Slight drift (0.1)')
ax1.axhline(y=0.2, color='red', linestyle='--', alpha=0.6, label='Significant drift (0.2)')
ax1.set_ylabel('PSI Score')
ax1.set_xlabel('Time Window')

ax2 = ax1.twinx()
ax2.plot(failure_rate['window'], failure_rate['failure_rate_pct'],
         'r-o', linewidth=2, label='Failure Rate %')
ax2.set_ylabel('Failure Rate (%)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

ax1.set_title('Drift (PSI) vs Failure Rate per Window')
plt.tight_layout()
plt.savefig('drift_vs_failure.png', dpi=150, bbox_inches='tight')
plt.show()
print("drift_vs_failure.png saved.")

In [ ]:
#Failure Prediction Model (Random Forest)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

features = ['avg_cpu_usage', 'avg_memory_usage', 'max_cpu_usage',
            'assigned_memory', 'scheduling_class', 'time_window']

X = df_pd[features].fillna(0)
y = df_pd['failed'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("=== Random Forest — Failure Prediction ===")
print(classification_report(y_test, y_pred, target_names=['No Failure', 'Failure']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

# Feature importance plot
importances = pd.Series(clf.feature_importances_, index=features).sort_values()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

importances.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importances')
axes[0].set_xlabel('Importance Score')

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['No Failure', 'Failure'],
    cmap='Blues',
    ax=axes[1]
)
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.savefig('failure_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print("failure_prediction.png saved.")

## Validator - Synthetic Drift Injection

In [ ]:
#Synthetic drift injection; validates the detector works

baseline = windows[1]["avg_cpu_usage"].dropna().values

# Inject drift by shifting mean by 2 standard deviations
drift_magnitude = 2 * baseline.std()
drifted = baseline + drift_magnitude

# Run KS test, should detect drift
stat, pvalue = stats.ks_2samp(baseline, drifted)
psi_score = compute_psi(baseline, drifted)

print("=== Synthetic Drift Injection Test ===")
print(f"Injected drift magnitude: +{drift_magnitude:.4f} (2 std devs)")
print(f"KS Statistic: {stat:.4f}")
print(f"KS p-value: {pvalue:.6f} → Drift detected: {pvalue < 0.05}")
print(f"PSI Score: {psi_score} → {('significant' if psi_score > 0.2 else 'slight' if psi_score > 0.1 else 'none')}")
print("\nValidator: If both detect drift here, the detector is working correctly.")

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(len(metrics), 2, figsize=(16, 5 * len(metrics)))

for i, metric in enumerate(metrics):
    # KS statistic over windows
    metric_ks = ks_df[ks_df["metric"] == metric]
    axes[i, 0].plot(metric_ks["window"], metric_ks["ks_statistic"],
                     marker='o', color='steelblue', linewidth=2)
    axes[i, 0].axhline(y=0.20, color='red', linestyle='--',
                    label='KS drift threshold = 0.20', alpha=0.7)
    axes[i, 0].fill_between(metric_ks["window"], metric_ks["ks_statistic"],
                          0.20, where=metric_ks["ks_statistic"] > 0.20,
                          alpha=0.2, color='red', label='drift zone')
    axes[i, 0].set_title(f"KS Statistic — {metric}")
    axes[i, 0].set_xlabel("Window")
    axes[i, 0].set_ylabel("KS Statistic")
    axes[i, 0].legend()

    # PSI over windows
    metric_psi = psi_df[psi_df["metric"] == metric]
    colors = ['green' if x == 'none' else 'orange' if x == 'slight' else 'red'
              for x in metric_psi["drift_level"]]
    axes[i, 1].bar(metric_psi["window"], metric_psi["psi"], color=colors)
    axes[i, 1].axhline(y=0.1, color='orange', linestyle='--',
                        label='slight drift (0.1)', alpha=0.7)
    axes[i, 1].axhline(y=0.2, color='red', linestyle='--',
                        label='significant drift (0.2)', alpha=0.7)
    axes[i, 1].set_title(f"PSI Score — {metric}")
    axes[i, 1].set_xlabel("Window")
    axes[i, 1].set_ylabel("PSI")
    axes[i, 1].legend()

plt.suptitle("Behavioral Drift Detection — Google Borg Cluster Traces",
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("drift_detection_results.png", dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")

## Streaming Simulation

In [ ]:
import time
import datetime

print("=" * 60)
print("SIMULATED STREAMING DRIFT DETECTION")
print("Processing windows sequentially as if data arrives live...")
print("=" * 60)

# Window 1 is our baseline
baseline_windows = {metric: windows[1][metric].dropna().values for metric in metrics}

# Warm-up: observe windows 2 and 3 to learn normal PSI variation
print("\n[WARM-UP] Learning normal PSI variation from windows 2-3...")
warmup_psi_scores = {metric: [] for metric in metrics}

for w in [2, 3]:
    for metric in metrics:
        current = windows[w][metric].dropna().values
        psi = compute_psi(baseline_windows[metric], current)
        warmup_psi_scores[metric].append(psi)

# Compute dynamic threshold per metric from warm-up
learned_thresholds = {}
for metric in metrics:
    scores = warmup_psi_scores[metric]
    mean_psi = np.mean(scores)
    std_psi = np.std(scores)
    learned_thresholds[metric] = {
        "slight": round(mean_psi + 1 * std_psi, 4),
        "significant": round(mean_psi + 2 * std_psi, 4)
    }
    print(f"  {metric}: slight threshold={learned_thresholds[metric]['slight']}, "
          f"significant threshold={learned_thresholds[metric]['significant']}")

print("\n[STREAMING] Processing windows 4-10 with learned thresholds...")
print("=" * 60)

alerts = []

for w in range(4, 11):
    time.sleep(0.5)

    timestamp = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"\n[{timestamp}] 📡 Window {w} received — analyzing...")

    window_drifted = False

    for metric in metrics:
        current = windows[w][metric].dropna().values

        ks_stat, pvalue = stats.ks_2samp(baseline_windows[metric], current)
        psi = compute_psi(baseline_windows[metric], current)

        # Use learned thresholds — no hardcoded values
        slight_thresh = learned_thresholds[metric]["slight"]
        sig_thresh = learned_thresholds[metric]["significant"]

        if psi >= sig_thresh:
            drift_severity = "significant"
        elif psi >= slight_thresh:
            drift_severity = "slight"
        else:
            drift_severity = "none"

        drift_detected = drift_severity != "none"

        if drift_detected:
            window_drifted = True
            alert = {
                "timestamp": timestamp,
                "window": w,
                "metric": metric,
                "ks_stat": ks_stat,
                "psi": psi,
                "severity": drift_severity
            }
            alerts.append(alert)

            if drift_severity == "significant":
                flag = "🔴 SIGNIFICANT DRIFT"
            elif drift_severity == "slight":
                flag = "🟡 SLIGHT DRIFT"
            else:
                flag = "🟠 DRIFT DETECTED"

            print(f"   {flag} | {metric} | KS={ks_stat:.4f} | PSI={psi} "
                  f"(threshold={slight_thresh})")

    if not window_drifted:
        print(f"   ✅ No drift detected — system stable")

print("\n" + "=" * 60)
print("STREAMING SIMULATION COMPLETE")
print(f"Total alerts raised: {len(alerts)}")
print("=" * 60)

if alerts:
    alerts_df = pd.DataFrame(alerts)
    print("\n=== Alert Summary Table ===")
    summary = alerts_df.groupby(['metric', 'severity']).size().reset_index(name='alert_count')
    summary = summary.sort_values('alert_count', ascending=False)
    print(summary.to_string(index=False))

    print("\n=== Most Affected Windows ===")
    print(alerts_df.groupby('window').size().reset_index(name='alerts_raised').to_string(index=False))
else:
    print("No alerts raised during simulation.")

In [ ]:
if alerts:
    alerts_df = pd.DataFrame(alerts)

    print("\n=== ALERT LOG ===")
    print(alerts_df.to_string(index=False))

    print("\n=== MOST AFFECTED METRICS ===")
    print(alerts_df.groupby("metric")["ks_stat"].mean().sort_values(ascending=False))

    print("\n=== DRIFT TIMELINE ===")
    print(alerts_df.groupby("window")["metric"].count().rename("metrics_drifted"))
else:
    print("No alerts raised.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 6))

severity_colors = {"none": "orange", "slight": "gold", "significant": "red"}

for alert in alerts:
    color = severity_colors.get(alert["severity"], "orange")
    y_pos = metrics.index(alert["metric"])
    ax.scatter(alert["window"], y_pos, color=color, s=alert["ks_stat"]*1000,
               alpha=0.7, edgecolors="black", linewidth=0.5)

ax.set_yticks(range(len(metrics)))
ax.set_yticklabels(metrics)
ax.set_xticks(range(2, 11))
ax.set_xlabel("Time Window (simulated stream)")
ax.set_ylabel("Metric")
ax.set_title("Real-Time Drift Alert Timeline — Google Borg Cluster Traces")
ax.axvline(x=1, color="green", linestyle="--", alpha=0.5, label="Baseline window")
ax.grid(True, alpha=0.3)

red_patch = mpatches.Patch(color='red', label='Significant drift')
gold_patch = mpatches.Patch(color='gold', label='Slight drift')
orange_patch = mpatches.Patch(color='orange', label='Drift detected')
ax.legend(handles=[red_patch, gold_patch, orange_patch])

plt.tight_layout()
plt.savefig("streaming_drift_alerts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Alert timeline saved.")

## Summary

In [ ]:
print("=" * 60)
print("BEHAVIORAL DRIFT DETECTION — FINAL SUMMARY")
print("=" * 60)

print("\n📊 Dataset: Google Borg Cluster Traces")
print(f"   Total records analyzed: {df.count():,}")
print(f"   Time windows: 10")
print(f"   Metrics monitored: {len(metrics)}")

print("\n🔍 KS Test — Drift detected in:")
drifted_ks = ks_df[ks_df["drift_detected"] == True]
for _, row in drifted_ks.iterrows():
    print(f"   Window {row['window']} | {row['metric']} | KS={row['ks_statistic']} | p={row['p_value']}")

print("\n📈 PSI — Significant drift in:")
drifted_psi = psi_df[psi_df["drift_level"] == "significant"]
for _, row in drifted_psi.iterrows():
    print(f"   Window {row['window']} | {row['metric']} | PSI={row['psi']}")

print("\n✅ Synthetic drift injection: VALIDATED")
print("=" * 60)